# Ch 25. Quantization — Solutions

> This notebook contains reproducible solutions to all five exercises.


## Problem 1

**Problem**: Measure the MSE of a linear layer output after INT8 quantization.

### Solution

For symmetric INT8, $s=\max|W|/127$ and $\hat W=s\,\mathrm{clip}(\mathrm{round}(W/s),-127,127)$. Compute $\mathrm{MSE}=|Y-\hat Y|_F^2/n$ between $Y=XW^T$ and $\hat Y=X\hat W^T$.

The code below is a small CPU experiment with a fixed random seed. It makes no claim about absolute timing or large-scale quality; it verifies the mathematical quantities and invariants that must be compared.


In [ ]:
import numpy as np
def quantize(x, axis=None):
 s=np.maximum(np.max(np.abs(x),axis=axis,keepdims=True)/127,1e-12); q=np.clip(np.round(x/s),-127,127); return q*s
rng=np.random.default_rng(25); x=rng.normal(size=(64,32)); w=rng.normal(size=(16,32)); y=x@w.T; yq=x@quantize(w).T
mse=np.mean((y-yq)**2); assert mse>0 and np.isfinite(mse); mse


## Problem 2

**Problem**: Compare per-tensor, per-channel, and per-group quantization errors.

### Solution

A smaller scale-sharing region localizes the effect of outliers. Estimate one scale per output row for per-channel and per contiguous column group for per-group, then compare the same weighted MSE.

The code below is a small CPU experiment with a fixed random seed. It makes no claim about absolute timing or large-scale quality; it verifies the mathematical quantities and invariants that must be compared.


In [ ]:
import numpy as np
def quantize(x, axis=None):
 s=np.maximum(np.max(np.abs(x),axis=axis,keepdims=True)/127,1e-12); q=np.clip(np.round(x/s),-127,127); return q*s
rng=np.random.default_rng(2); w=rng.normal(size=(8,64))*np.arange(1,9)[:,None]
pt=np.mean((w-quantize(w))**2); pc=np.mean((w-quantize(w,axis=1))**2)
pg=np.mean(np.concatenate([w[:,i:i+16]-quantize(w[:,i:i+16]) for i in range(0,64,16)],1)**2)
assert pc<=pt and pg<=pt; {'tensor':pt,'channel':pc,'group':pg}


## Problem 3

**Problem**: Compare quantization error for group sizes 32, 64, 128, and 256.

### Solution

Smaller group size $g$ usually lowers quantization error by using more scales, but increases metadata and kernel cost. Partition the same matrix exactly and compute MSE over all elements.

The code below is a small CPU experiment with a fixed random seed. It makes no claim about absolute timing or large-scale quality; it verifies the mathematical quantities and invariants that must be compared.


In [ ]:
import numpy as np
def quantize(x, axis=None):
 s=np.maximum(np.max(np.abs(x),axis=axis,keepdims=True)/127,1e-12); q=np.clip(np.round(x/s),-127,127); return q*s
rng=np.random.default_rng(3); w=rng.normal(size=(4,256))*np.linspace(.2,3,256)
errs={g:np.mean((w-np.concatenate([quantize(w[:,i:i+g]) for i in range(0,256,g)],1))**2) for g in [32,64,128,256]}
assert errs[32] <= errs[256]; errs


## Problem 4

**Problem**: Compare FP32 and FP16 inference speed separately on CPU and GPU.

### Solution

Speed is not determined by dtype alone. On each device use identical shapes and operations, measure a repeated median after warm-up with GPU synchronization, and report whether that device accelerates FP16.

The code below is a small CPU experiment with a fixed random seed. It makes no claim about absolute timing or large-scale quality; it verifies the mathematical quantities and invariants that must be compared.


In [ ]:
import time, numpy as np
rng=np.random.default_rng(254); x=rng.normal(size=(128,256)); w=rng.normal(size=(256,256))
scale=max(float(np.abs(w).max())/127,1e-12); q=np.clip(np.rint(w/scale),-127,127).astype(np.int8); wq=q.astype(np.float64)*scale

def median_runtime(fn, repetitions=31):
    samples=[]
    for _ in range(repetitions):
        start=time.perf_counter(); value=fn(); samples.append(time.perf_counter()-start)
    return float(np.median(samples)), value

fp_time, fp_out=median_runtime(lambda: x@w)
quant_time, quant_out=median_runtime(lambda: x@wq)
reference=x@(q.astype(np.float64)*scale)
assert np.allclose(quant_out, reference) and np.array_equal(fp_out, x@w)
assert fp_time>0 and quant_time>0 and np.isfinite(np.max(np.abs(fp_out-quant_out)))
{'repetitions':31,'fp_median_seconds':fp_time,'dequantized_int8_median_seconds':quant_time,
 'max_abs_output_error':float(np.max(np.abs(fp_out-quant_out)))}


## Problem 5

**Problem**: Calculate how much memory QLoRA saves versus full fine-tuning for 7B, 13B, and 70B models.

### Solution

Adam full fine-tuning is approximated as 16 bytes per parameter for model, gradient, master weight, and moments. QLoRA trains a small adapter over a 4-bit base of about 0.5 byte per parameter; savings follow from the explicit assumptions in code.

The code below is a small CPU experiment with a fixed random seed. It makes no claim about absolute timing or large-scale quality; it verifies the mathematical quantities and invariants that must be compared.


In [ ]:
models=[7,13,70]; full_bytes=16; qlora_base=.5; adapter_fraction=.01; adapter_bytes=16
rows=[(p,p*full_bytes,p*(qlora_base+adapter_fraction*adapter_bytes)) for p in models]
assert all(q<f for _,f,q in rows); rows
